In [17]:
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE

In [18]:
df = pd.read_csv("cleaned_dataset.csv")

In [19]:
X = df.drop("Accident_severity", axis=1)
y = df["Accident_severity"]

# Encode target only
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)

In [20]:
joblib.dump(target_encoder, "target_encoder.pkl")

['target_encoder.pkl']

In [27]:
# Categorical features
categorical_features = X.select_dtypes(include='object').columns

# One-hot encode directly
X = pd.get_dummies(
    X,
    columns=categorical_features
)

unknown_cols = [col for col in X.columns if "unknown" in col.lower()]
X = X.drop(columns=unknown_cols)

# Feature selection
selector = SelectKBest(
    score_func=chi2,
    k=60
)

X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

print("Selected Features:")
print(selected_features)

Selected Features:
Index(['Number_of_vehicles_involved', 'Number_of_casualties',
       'Day_of_week_friday', 'Day_of_week_monday', 'Day_of_week_saturday',
       'Age_band_of_driver_31-50', 'Age_band_of_driver_over 51',
       'Age_band_of_driver_under 18', 'Vehicle_driver_relation_other',
       'Vehicle_driver_relation_owner', 'Driving_experience_Intermediate',
       'Driving_experience_New Driver', 'Type_of_vehicle_Heavy Vehicle',
       'Type_of_vehicle_Other', 'Type_of_vehicle_Special Vehicle',
       'Owner_of_vehicle_governmental', 'Owner_of_vehicle_organization',
       'Owner_of_vehicle_other', 'Service_year_of_vehicle_1-2yr',
       'Service_year_of_vehicle_5-10yrs', 'Service_year_of_vehicle_below 1yr',
       'Defect_of_vehicle_7', 'Area_accident_occured_Commercial/Workplace',
       'Area_accident_occured_Public/Institutional',
       'Area_accident_occured_Rural',
       'Area_accident_occured_Urban/Residential',
       'Lanes_or_Medians_double carriageway (median)',
   

In [28]:
joblib.dump(selected_features, "selected_features.pkl")

['selected_features.pkl']

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [30]:
smote = SMOTE(random_state=42)

X_train, y_train = smote.fit_resample(
    X_train,
    y_train
)

In [31]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [32]:
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [33]:
joblib.dump(X_train, "X_train.pkl")
joblib.dump(X_test, "X_test.pkl")
joblib.dump(y_train, "y_train.pkl")
joblib.dump(y_test, "y_test.pkl")

['y_test.pkl']